# Hilbert Transform and Amplitude Envelope

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

The Hilbert transform extracts the instantaneous amplitude envelope from the signal. We build an analytic signal from the real signal, and its absolute value gives the instantaneous amplitude envelope.

## Expected outputs

- A red envelope tracking the signal peaks on both sides
- High-envelope regions indicate periods of intense activity
- Low-envelope regions indicate periods of calm

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Channel | P4 | Parietal region |
| Sampling rate | 200 Hz | One sample every 5 ms |
| Plotted samples | 5000 | First 25 seconds |


## 1. Install dependencies


In [ ]:
!pip install scipy numpy plotly wfdb pywt


## 2. Clone repo and download data

We download only subject 1 (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2, channel P4 (parietal region).


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


## 4. Apply the Hilbert transform

We use `scipy.signal.hilbert` to build the complex analytic signal, then extract the absolute value to compute the instantaneous amplitude envelope.


In [ ]:
from scipy.signal import hilbert

n_plot = min(5000, len(channel_data))
signal = channel_data[:n_plot]

analytic_signal = hilbert(signal)
amplitude_envelope = np.abs(analytic_signal)
print(f'Envelope range: {amplitude_envelope.min():.2f} - {amplitude_envelope.max():.2f} uV')


## 5. Interactive plot

**What to look for:**

- The red envelope surrounds the signal on both sides
- High-envelope regions indicate intense activity
- Use the zoom tool to inspect specific time ranges



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

t_sec = np.arange(n_plot) / fs

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original signal - Channel P4',
                                    'Signal with amplitude envelope - Channel P4'))
fig.add_trace(go.Scatter(x=t_sec, y=signal, name='Signal',
                         line=dict(color='blue', width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=signal, name='Signal',
                         line=dict(color='blue', width=0.5, opacity=0.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=amplitude_envelope, name='Envelope',
                         line=dict(color='red', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=-amplitude_envelope, name='-Envelope',
                         line=dict(color='red', width=1.5)), row=2, col=1)
fig.update_layout(height=700, title_text='Hilbert Transform - Amplitude Envelope - Channel P4',
                  xaxis_title='Time (s)', xaxis2_title='Time (s)',
                  yaxis_title='Amplitude (uV)', yaxis2_title='Amplitude (uV)',
                  showlegend=False)
fig.show()


## What did we learn?

- The Hilbert transform extracts the instantaneous amplitude envelope from the signal
- The envelope tracks the signal peaks and summarizes its instantaneous energy
- It is used to measure brain activity power in a specific frequency band over time
- It is also used in brain connectivity studies to measure synchronization between regions

